# 📗 세 검색 경로를 통합하고 멀티홉 근거로 답변합니다

## 이번 실습에서 무엇을 만들까요?

**관계 조건과 줄거리 조건을 함께 만족하는 영화를 찾고, 근거를 인용해 답하는 검색 흐름**을 만듭니다.

> The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중,
> 뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?

이 질문에는 **배우·개봉연도 조건**과 **줄거리의 의미**가 함께 들어 있습니다.
전문·벡터 검색은 줄거리가 관련 있는 문서를 찾고, 그래프 검색은 배우·연도 조건의 후보와 출연 근거를 가져옵니다.
세 결과를 같은 원문 ID로 연결하고 관계 후보만 남긴 뒤, 원문과 근거를 읽어 답변합니다.

<img src="images/10_lesson02_purpose.png" width="1100" alt="배우·연도와 줄거리 조건을 가진 질문을 전문·벡터·그래프 검색으로 나눕니다. 문서 ID로 결합하고 관계 후보를 유지한 뒤 PageRank 순서를 비교하고 출연·줄거리·연도 근거를 인용해 답변합니다.">

## 이번 교안에서 배울 내용

1. Text-to-Cypher로 관계 후보·출연 근거를 가져오고, 전문·벡터 결과와 원문 ID로 결합합니다.
2. 같은 후보에서 PageRank 비중을 바꾸며 줄거리 관련성이 유지되는지 비교합니다.
3. 줄거리·공통 배우·개봉연도를 실제 근거와 연결해 답변하고 인용을 확인합니다.


## 준비한 영화 자료를 연결합니다

이전 교안에서 사용한 **Neo4j Movies 그래프 171개 개체·253개 관계, 영화 원문 38편,
원문 청크 571개와 768차원 배포 벡터**를 그대로 사용합니다. 영화 관계는 Neo4j 샘플,
줄거리는 버전이 고정된 Wikipedia 원문입니다. 두 출처를 구분해서 인용합니다.

노트북 폴더에서 시작하며, 패키지·DB 설정은 README를 참고하세요.

검색·그래프·모델에 필요한 라이브러리를 불러옵니다.


In [ ]:
# 검색·DB·모델에 필요한 라이브러리를 불러옵니다.
import json
import os
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field


자료 경로를 지정하고 그래프·벡터 파일을 읽는 함수를 불러옵니다.


In [ ]:
# 노트북 기준 경로입니다. 정답은 상위 폴더를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
sys.path.insert(0, str(material_dir.resolve()))
from graph_data import load_graph, read_vectors


`.env`로 Neo4j에 연결하고 임베딩·답변 모델과 Cypher 실행 함수를 준비합니다.


In [ ]:
# 공통 설정을 읽고 교안의 .env를 우선 적용합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# 모델만 준비합니다. API 호출은 검색·답변 단계에서 합니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 준비한 청크를 자동으로 나누지 않습니다.
)
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
address = urlparse(neo4j_uri)
print("Neo4j 연결:", address.hostname, address.port or 7687)


def run_cypher(query, **params):
    """Cypher 결과를 행 딕셔너리의 리스트로 반환합니다."""
    # 인자를 Cypher의 $파라미터로 전달합니다.
    with driver.session() as session:
        # 조회 결과를 딕셔너리 목록으로 반환합니다.
        return [record.data() for record in session.run(query, **params)]


영화 그래프·원문·기존 임베딩을 불러오고 자료 규모를 확인합니다.


In [ ]:
# 영화 그래프·원문과 기존 청크 벡터를 불러옵니다.
movies = load_graph(data_dir / "movies_extraction_packet.json")
chunk_ids, vectors = read_vectors(
    data_dir / "embeddings/movies_complete.npz", movies["chunks"], embedding_model
)
# 검색 결과의 원문 ID로 줄거리를 찾을 사전입니다.
documents_by_id = movies["documents"]
print("개체:", len(movies["nodes"]), "관계:", len(movies["relations"]))
print("문서:", len(documents_by_id), "청크:", len(chunk_ids), "벡터:", vectors.shape)


### 영화 그래프와 원문·청크를 적재합니다

원래 ID·속성·관계·출처를 유지하며 `Movie`, `Person`, `Document`, `Chunk`에 저장합니다.
`RAGEntity`는 영화·인물의 공통 ID 조회와 고유 제약을 위해 추가하는 레이블입니다.

ID 중복 방지 제약을 만들고 영화·인물·관계를 Neo4j에 저장합니다.


In [ ]:
# 개체의 standard_id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT rag_entity_id IF NOT EXISTS FOR (n:RAGEntity) REQUIRE n.standard_id IS UNIQUE")
# 원문 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT document_id IF NOT EXISTS FOR (n:Document) REQUIRE n.id IS UNIQUE")
# 청크 id 중복을 막는 고유 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE")

# 영화·인물을 종류별로 적재합니다.
for entity_type in ["Movie", "Person"]:
    # 현재 종류의 개체만 고릅니다.
    rows = [row for row in movies["nodes"] if row["entity_type"] == entity_type]
    # 현재 종류의 영화·인물 노드를 만들고 속성을 갱신합니다.
    run_cypher(f"""
        // 목록을 개체별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 같은 ID의 노드를 재사용하고, 없으면 생성합니다.
        MERGE (n:RAGEntity {{standard_id: item.standard_id}})
        // 종류 레이블을 추가하고 속성·이름을 갱신합니다.
        SET n:{entity_type}, n += item.properties, n.name = item.name
    """, rows=rows)

# 적재한 노드 사이에 종류별 관계를 연결합니다.
for relation in ["ACTED_IN", "DIRECTED", "PRODUCED", "WROTE", "REVIEWED", "FOLLOWS"]:
    # 현재 유형의 관계만 고릅니다.
    rows = [row for row in movies["relations"] if row["relation"] == relation]
    # 현재 유형의 관계를 연결하고 근거·출처를 저장합니다.
    run_cypher(f"""
        // 목록을 관계별 행으로 펼칩니다.
        UNWIND $rows AS item
        // 관계의 시작·끝 노드를 ID로 찾습니다.
        MATCH (a:RAGEntity {{standard_id: item.subject_id}})
        MATCH (b:RAGEntity {{standard_id: item.object_id}})
        // 같은 유형·방향·근거 ID의 관계가 없으면 만들고, 있으면 재사용합니다.
        MERGE (a)-[r:{relation} {{claim_id: item.claim_id}}]->(b)
        // 관계 속성과 근거·출처를 저장합니다.
        SET r += item.properties, r.evidence = item.evidence,
            r.source_url = item.source_url, r.source_doc_id = item.source_doc_id
    """, rows=rows)


원문·청크·기존 벡터를 영화에 연결하고 전문·벡터 검색 인덱스를 만듭니다.


In [ ]:
# 원문 노드를 먼저 저장합니다.
run_cypher("""
    // 원문 목록을 한 문서당 한 행으로 펼칩니다.
    UNWIND $rows AS item
    // 같은 ID의 원문 노드가 없으면 만들고, 있으면 재사용합니다.
    MERGE (d:Document {id: item.doc_id})
    // 제목·본문·출처 URL을 갱신합니다.
    SET d.title = item.title, d.text = item.text, d.url = item.url
""", rows=list(documents_by_id.values()))

# 원문과 해당 영화를 연결합니다.
run_cypher("""
    // doc_id로 원문을, standard_id로 해당 영화를 찾습니다.
    UNWIND $rows AS item
    MATCH (d:Document {id: item.doc_id})
    MATCH (m:Movie {standard_id: item.standard_id})
    // 원문과 해당 영화를 ABOUT_MOVIE 관계로 연결합니다.
    MERGE (d)-[:ABOUT_MOVIE]->(m)
""", rows=movies["document_links"])

# 청크와 벡터의 순서를 맞춰 전체 저장 목록을 만듭니다.
rows = []
for index, chunk in enumerate(movies["chunks"]):
    # NumPy 벡터를 DB에 전달할 목록으로 바꿉니다.
    rows.append({"id": chunk_ids[index], "doc_id": chunk.metadata["source_doc_id"],
                 "text": chunk.page_content, "embedding": vectors[index].tolist()})

# 전체 청크와 벡터를 한 번에 저장합니다.
run_cypher("""
    UNWIND $rows AS item
    // 원문을 찾고 청크를 ID로 생성·재사용합니다.
    MATCH (d:Document {id: item.doc_id})
    MERGE (c:Chunk {id: item.id})
    // 이 교안의 영화 청크만 검색하도록 전용 레이블을 붙입니다.
    SET c:Day49MovieChunk
    // text는 전문 검색, embedding은 벡터 검색, source_id는 원문별 결과 통합에 씁니다.
    SET c.text = item.text, c.source_id = item.doc_id, c.embedding = item.embedding
    // 청크와 원문을 FROM_DOCUMENT 관계로 연결합니다.
    MERGE (c)-[:FROM_DOCUMENT]->(d)
""", rows=rows)

# 이전 버전이 전체 Chunk를 색인했다면 이 교안의 두 인덱스만 다시 만듭니다.
index_properties = {"movie_chunks": ["embedding"], "movie_fulltext": ["text"]}
existing_indexes = run_cypher("""
    SHOW INDEXES YIELD name, labelsOrTypes, properties
    WHERE name IN ['movie_chunks', 'movie_fulltext']
    RETURN name, labelsOrTypes, properties
""")
for index in existing_indexes:
    if index["labelsOrTypes"] != ["Day49MovieChunk"] or index["properties"] != index_properties[index["name"]]:
        run_cypher(f"DROP INDEX `{index['name']}` IF EXISTS")

# 벡터 유사도 검색용 인덱스를 만듭니다.
run_cypher("""
    CREATE VECTOR INDEX movie_chunks IF NOT EXISTS
    // 이 교안의 영화 청크만 벡터 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON c.embedding
    // 저장된 벡터의 768차원과 코사인 유사도를 설정합니다.
    OPTIONS {indexConfig: {`vector.dimensions`: 768, `vector.similarity_function`: 'cosine'}}
""")
# 벡터 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_chunks', 300)")
# 단어 일치 검색용 전문 인덱스를 만듭니다.
run_cypher("""
    CREATE FULLTEXT INDEX movie_fulltext IF NOT EXISTS
    // 같은 영화 청크의 text만 전문 검색 대상으로 지정합니다.
    FOR (c:Day49MovieChunk) ON EACH [c.text]
    // 영어 분석기를 사용하고, 본문 변경을 커밋 시 인덱스에 반영합니다.
    OPTIONS {indexConfig: {`fulltext.analyzer`: 'english', `fulltext.eventually_consistent`: false}}
""")
# 전문 인덱스가 검색 가능한 상태가 될 때까지 최대 300초 기다립니다.
run_cypher("CALL db.awaitIndex('movie_fulltext', 300)")
print("기존 문서 임베딩을 적재했습니다. 문서 임베딩 API 호출은 없습니다.")


### 적재된 그래프 전체 구조

청크·원문·영화·인물의 연결과 검색 인덱스를 한눈에 봅니다.

<img src="images/06_loaded_graph_v2.png" width="1000" alt="적재된 영화 그래프: 청크에서 원문과 영화로 연결되며, 인물은 영화 및 다른 인물과 관계를 맺습니다. 검색 인덱스는 청크에 적용됩니다.">


한 영화의 여러 청크가 함께 검색될 수 있습니다. 이번 실습은 영화별로 순위를 합치므로, 검색된 청크 중 영화마다 최고 점수 한 건을 대표로 남깁니다. 청크 자체를 답변 근거로 사용할 때는 같은 영화의 여러 청크를 함께 유지할 수도 있습니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


def show_documents(rows):
    """전달받은 모든 검색 결과를 순서대로 조회해 제목과 본문 앞부분을 보여 줍니다."""
    display(pd.DataFrame(rows))
    # 검색 순위를 함께 전달해 DB 조회 후에도 같은 순서를 유지합니다.
    ranked_ids = [{"rank": rank, "source_id": row["source_id"]}
                  for rank, row in enumerate(rows)]
    # 전달받은 원문 전체의 제목과 본문을 Neo4j에서 한 번에 조회합니다.
    documents = run_cypher("""
        UNWIND $rows AS item
        // 검색 결과의 원문 ID로 저장된 문서를 찾습니다.
        MATCH (d:Document {id: item.source_id})
        RETURN d.id AS source_id, d.title AS title, d.text AS text
        // DB의 반환 순서를 원래 검색 순위에 맞춥니다.
        ORDER BY item.rank
    """, rows=ranked_ids)
    for doc in documents:
        print("\n제목:", doc["title"], "| 원문 ID:", doc["source_id"])
        # 긴 원문은 앞 650자만 출력합니다.
        print(doc["text"][:650])


## 1. 줄거리와 관계 조건을 나누어 검색합니다

| 검색 구조 | 사용하는 방법 | 그래프의 역할 |
|---|---|---|
| 교안01: 벡터 진입 뒤 확장 | `VectorCypherRetriever` | 진입 영화에서 출연 관계를 따라 후보를 늘립니다. |
| 교안02: 세 경로 결과 융합 | `GraphCypherQAChain` + 전문·벡터 검색 | 질문의 관계 조건을 만족하는 후보 집합을 별도로 찾습니다. |

두 교안은 같은 데이터로 서로 다른 결합 방식을 구현합니다. 각각 준비 셀부터 독립적으로 시작합니다.
이번에는 학습자가 `semantic_query`·`graph_request`를 정하고 검색·재정렬·답변을 셀 순서대로 연결합니다.

**질문:** The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중,
뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?

| 질문의 요구 | 사용할 입력 | 확인할 결과 |
|---|---|---|
| 뇌에 기밀 데이터를 운반하는 이야기 | 한글 줄거리 검색어 | 전문 검색·Dense 후보와 원문 |
| The Matrix와 배우 공유 | 출연 관계 두 개 | 공통 배우와 양쪽 관계 ID |
| 1999년 이전 | `released < 1999` | 개봉연도 |

줄거리 검색어와 배우·연도 조건은 직접 나눕니다. 벡터 검색은 한글 질문을 사용하고,
전문 검색은 Jev로 언어를 판단한 뒤, 한글은 구조화된 영어 검색어로 바꾸고 영어는 그대로 사용합니다.

전체 질문을 줄거리 검색어와 배우·연도 조건으로 나누고 질문 벡터를 만듭니다.


In [ ]:
question = "The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중, 뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?"
semantic_query = "기밀 정보를 뇌 속 장치에 저장해 운반하는 사람이 위험에 빠지는 영화"
# 질문 벡터를 한 번 만들어 재사용합니다.
query_vector = embedding_model.embed_query(semantic_query)


### GraphCypherQAChain으로 관계 후보를 가져옵니다

**Text-to-Cypher**는 자연어 질문을 그래프 DB 조회 언어인 **Cypher 쿼리로 변환**하는 작업입니다.
**GraphCypherQAChain**은 LLM으로 이 변환을 수행하고, 생성한 쿼리를 Neo4j에서 실행하는 체인입니다.

이 실습의 흐름: **자연어 질문 → LLM의 Cypher 생성 → Neo4j 조회 → 조회 결과 반환**

- `return_direct=True`: LLM 답변 생성 없이 DB 조회 결과를 반환합니다.
- `return_intermediate_steps=True`: 생성한 Cypher 등 중간 실행 정보를 함께 반환합니다.

이번 자료의 영화는 38편이므로 `LIMIT 100`과 `top_k=100`으로 관계 조건 집합의 일부가 임의로 잘리는 것을 피합니다.

이 절에는 APOC가 필요하며, 앞에서 설정한 `NEO4J_USER`·`NEO4J_PASSWORD`로 같은 실습 DB에 연결합니다.
**학습 데이터만 있는 독립 실습 DB**에서 진행합니다.
`allow_dangerous_requests=True`도 권한 제한 기능이 아닙니다. 생성 쿼리의 내용을 반드시 읽으세요.
[GraphCypherQAChain 공식 문서](https://docs.langchain.com/oss/python/integrations/graphs/neo4j_cypher)

한글 관계 질문을 Cypher로 바꿔 조회하는 `GraphCypherQAChain`을 구성합니다.


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# 앞에서 설정한 실습 DB 계정으로 연결합니다.
graph = Neo4jGraph(
    url=neo4j_uri,
    username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=False,  # 속성 예시 값 수집을 생략하고 기본 스키마를 사용합니다.
)
# 관계 질문을 지정한 스키마의 조회 Cypher로 바꾸는 규칙을 정합니다.
# GraphCypherQAChain이 {schema}에는 DB에서 읽어 include_types로 추린 구조를,
# {question}에는 invoke에 전달한 질문을 자동으로 넣습니다.
cypher_prompt = PromptTemplate.from_template("""
다음 스키마만 사용해 조회용 Cypher 하나를 작성하세요.
{schema}

사용할 관계 경로:
(base:Movie)<-[first:ACTED_IN]-(actor:Person)-[second:ACTED_IN]->(candidate:Movie)<-[:ABOUT_MOVIE]-(d:Document)

기준 영화명·개봉연도·최소 공유 배우 수는 질문에서 읽으세요.
한글과 영문 영화명이 함께 있으면 영문 title을 사용하세요.
기준 영화명은 base.title에, 개봉연도 조건은 candidate.released에만 적용하세요.
다른 영화를 요구하면 candidate <> base로 기준 영화를 제외하세요.
이 조건들은 집계 WITH 앞의 WHERE에서 적용하세요.

공유 배우와 두 출연 관계의 실제 claim_id는 다음처럼 함께 집계하세요.
WITH candidate, d, count(DISTINCT actor) AS shared_count,
     collect(DISTINCT {{actor: actor.name, first_claim: first.claim_id, second_claim: second.claim_id}}) AS paths
공유 배우 수 조건은 이 WITH 뒤의 WHERE에서 적용하세요.

반환 형식:
RETURN d.id AS source_id, candidate.title AS title, candidate.released AS released,
       candidate.standard_id AS movie_id, shared_count, paths
ORDER BY source_id LIMIT 100

질문에 없는 조건이나 줄거리 속성을 추가하지 마세요. Cypher만 반환하세요.
질문: {question}
""")
# 질문 → Cypher 생성 → DB 조회를 연결합니다.
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    # 필요한 레이블·관계 유형만 프롬프트에 넣습니다.
    include_types=["Movie", "Person", "Document", "ACTED_IN", "ABOUT_MOVIE"],
    return_direct=True,  # LLM 답변 생성 없이 DB 조회 결과를 반환합니다.
    return_intermediate_steps=True,  # 생성한 Cypher 등 중간 실행 정보를 함께 반환합니다.
    top_k=100,  # 조회 결과를 최대 100행 사용합니다.
    # LLM 생성 쿼리의 실행 위험에 동의합니다. False이면 체인 생성이 차단됩니다.
    allow_dangerous_requests=True,
)


질문에 개봉연도와 기준 영화 제외 조건을 모두 넣습니다. 출력된 쿼리에서 `ACTED_IN` 방향과 연도 비교를 확인하세요.

한글 관계 질문으로 후보와 출연 근거를 조회하고 생성 Cypher를 확인합니다.


In [ ]:
graph_request = "매트릭스(The Matrix)와 배우를 최소 한 명 공유하며 1999년 이전에 개봉한 다른 영화를 모두 찾아 주세요."
# LLM이 관계 질문을 Cypher로 바꿔 실행하면, 생성된 쿼리를 출력합니다.
graph_result = graph_chain.invoke({"query": graph_request})
print(graph_result["intermediate_steps"][0]["query"])
graph_rows = graph_result["result"]
display(pd.DataFrame(graph_rows))


### 관계 검색 결과와 출연 근거를 확인합니다

생성 쿼리는 후보마다 원문 ID·제목·개봉연도·영화 ID·공유 배우 수·출연 경로를 반환합니다.
`paths`에는 공통 배우 이름과 두 영화의 출연 관계 ID가 들어 있습니다.
이 결과의 원문 ID는 통합 후보를 제한할 때, 출연 경로는 답변 근거를 모을 때 재사용합니다.

**질문의 조건은 Text-to-Cypher가 해석합니다.** 반환된 후보만 남겨도 그 해석이 맞다는 보장은 없습니다.
생성 Cypher의 기준 영화·연도·배우 수와 실제 반환 근거를 읽고, 잘못 해석한 부분이 있으면 질문이나 프롬프트를 고쳐 다시 조회하세요.

관계 후보 ID를 모으고 반환된 개봉연도·공통 배우·출연 경로를 확인합니다.


In [ ]:
# Text-to-Cypher가 찾은 후보 ID를 통합 검색의 관계 조건으로 사용합니다.
allowed_ids = {row["source_id"] for row in graph_rows}
# 후보 영화의 연도·공통 배우 수·양쪽 출연 근거를 확인합니다.
display(pd.DataFrame(graph_rows, columns=["title", "released", "shared_count", "paths"]))


`Johnny Mnemonic`과 `The Devil's Advocate`처럼 관계 조건이 맞아도 줄거리는 서로 다를 수 있습니다.
관계 후보의 줄거리가 질문에 맞는지는 전문·벡터 검색과 원문으로 확인합니다.


### 🖐️ 함께 따라하기: Apollo 13의 관계 후보와 출연 근거를 확인합니다

기준 영화 `Apollo 13`, 개봉연도 1999~2000, 공유 배우 최소 1명, 기준 영화 제외 조건을 자연어로 질문하세요. `practice_graph_rows`에 체인의 조회 결과를 담고 생성 Cypher·개봉연도·공통 배우·양쪽 출연 관계 ID를 읽으세요.

Apollo 13의 관계 조건을 자연어로 전달하고 생성 쿼리와 출연 근거를 확인합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

ID 오름차순으로 반환된 관계 결과에서 첫 영화를 가장 관련 있는 영화로 볼 수 있나요?

<details><summary>해설 보기</summary>

ID 정렬은 출력 순서만 고정합니다. 그래프는 후보와 출연 근거를 제공하며, 줄거리가 질문에 맞는지는 전문·벡터 검색과 원문으로 판단합니다.

</details>


## 2. 전문·벡터 검색을 합치고 관계 후보를 남깁니다

**전문 검색은 단어, 벡터 검색은 의미로 줄거리를 찾습니다.** 두 검색의 순위를 RRF로 합칩니다.
**그래프 검색은 배우·연도 조건의 후보와 출연 근거를 제공합니다.** 그래프 결과에는 RRF 점수를 매기지 않습니다.

<img src="images/11_ensemble_graph_flow.png" width="1100" alt="전문·벡터 검색은 EnsembleRetriever의 RRF로 합칩니다. 그래프 결과는 후보 제한과 누락 후보 보완 및 근거 수집에 사용합니다.">

### 어떤 결합 도구를 사용할까요?

| 도구 | 역할 |
|---|---|
| LangChain `EnsembleRetriever` | 여러 검색 결과를 가중 RRF로 결합합니다. 이번 교안에서 사용합니다. |
| Neo4j `HybridRetriever` | 전문·벡터 검색 점수를 정규화해 결합합니다. 설치된 버전의 방식은 RRF와 다릅니다. |
| Neo4j `HybridCypherRetriever` | 하이브리드 검색 결과에서 추가 Cypher로 관계를 탐색합니다. |

[EnsembleRetriever 공식 문서](https://reference.langchain.com/python/langchain-classic/retrievers/ensemble/EnsembleRetriever) · [Neo4j HybridRetriever](https://neo4j.com/docs/neo4j-graphrag-python/current/api.html#hybridretriever)

두 교안은 독립적으로 시작할 수 있습니다. 아래에서 `movie_fulltext`의 `ONLINE` 상태와 `english` 설정을 확인하세요.
`IF NOT EXISTS`는 기존 인덱스 설정을 변경하지 않습니다.

전문 인덱스의 준비 상태와 영어 분석기 설정을 확인합니다.


In [ ]:
# 인덱스 상태와 실제 분석기 설정을 확인합니다.
display(pd.DataFrame(run_cypher("""
    SHOW FULLTEXT INDEXES YIELD name, state, populationPercent, labelsOrTypes, options
    WHERE name = 'movie_fulltext'
    RETURN name, state, populationPercent, labelsOrTypes, options
""")))


언어에 맞는 검색어로 전문 검색하고, 원문별 최고 점수 청크를 반환하는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    print("질문 언어:", language, "| 전문 검색어:", search_text)
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


### EnsembleRetriever로 두 검색 순위를 합칩니다

가중 RRF는 각 문서의 검색 순위 `r`에 `가중치 / (60 + r)`를 적용해 합칩니다.
`EnsembleRetriever`가 이 계산과 정렬을 처리합니다. 전문 검색 비중은 0.4, 벡터 검색은 0.6입니다.

- `Document`: LangChain이 사용하는 문서 형식입니다. 원문 ID는 `metadata`에 넣습니다.
- `RunnableLambda`: 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
- `id_key="source_id"`: 두 검색에서 찾은 같은 원문을 한 문서로 합칩니다. 각 검색 내부의 원문 중복은 미리 제거합니다.

`ensemble_retriever.invoke(검색어)`는 두 검색부터 RRF 결합까지 실행합니다.
아래에서는 단독 검색과 비교할 수 있도록 각각 검색한 뒤, `weighted_reciprocal_rank`로 받은 결과만 합칩니다.

전문·벡터 검색을 Document 형식으로 연결하고 EnsembleRetriever를 준비합니다.


In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever


def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문·벡터 검색만 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[fulltext_retriever, dense_retriever],
    weights=[0.4, 0.6], c=60, id_key="source_id",
)


전문·벡터 검색 결과를 EnsembleRetriever의 가중 RRF로 결합합니다.


In [ ]:
fulltext_docs = fulltext_retriever.invoke(semantic_query)
# 앞에서 만든 질문 벡터를 재사용합니다.
dense_docs = as_documents(vector_documents(query_vector, top_k=30))
fused_docs = ensemble_retriever.weighted_reciprocal_rank([fulltext_docs, dense_docs])
# 반환 순위를 보관합니다. search_score는 RRF 점수가 아니라 순위의 역수입니다.
fused_rows = [{"source_id": doc.metadata["source_id"], "search_rank": rank, "search_score": 1 / rank}
              for rank, doc in enumerate(fused_docs, start=1)]
display(pd.DataFrame(fused_rows))


### 가져올 검색 결과 수와 답변에 쓸 영화 수를 구분합니다

`top_k=30`은 각 검색에서 가져올 **최대 청크 수**입니다.
같은 영화의 청크를 합치고 관계 조건에 맞는 후보만 남기면 영화 수는 줄어듭니다.
그중 **최대 3편의 원문과 근거**를 답변 모델에 전달합니다. 답변에는 질문에 맞는 작품만 사용합니다.
원하는 영화가 검색되지 않으면 검색어와 `top_k`를 확인하세요.

### 검색 순위와 관계 조건을 함께 적용합니다

줄거리가 비슷해도 배우·연도 조건이 다를 수 있습니다.
RRF 결과에서 `allowed_ids`에 있는 영화만 남깁니다.
그래프에서만 찾은 영화는 검색 점수 0으로 뒤에 추가해 원문을 함께 확인합니다.

관계 조건 밖의 영화를 제외하고 그래프에서만 찾은 후보를 추가합니다.


In [ ]:
# RRF 순서를 유지하면서 관계 조건 후보만 남깁니다.
candidate_rows = [row for row in fused_rows if row["source_id"] in allowed_ids]
removed_rows = [row for row in fused_rows if row["source_id"] not in allowed_ids]
# 전문·벡터 검색이 놓친 관계 후보는 검색 점수 0으로 뒤에 추가합니다.
fused_ids = {row["source_id"] for row in fused_rows}
graph_only_ids = sorted(allowed_ids - fused_ids)
candidate_rows += [{"source_id": source_id, "search_rank": None, "search_score": 0.0}
                   for source_id in graph_only_ids]
print("RRF 문서:", len(fused_rows), "조건 제외:", len(removed_rows), "그래프 추가:", len(graph_only_ids))
show_documents(candidate_rows)


### 단독 검색과 RRF 결과를 비교합니다

같은 검색 결과를 재사용해 전문 단독·벡터 단독·RRF의 상위 작품을 비교합니다.
각 목록에 같은 관계 후보 ID를 적용합니다. 그래프가 추가한 후보도 원문으로 확인하세요.

전문 단독·벡터 단독·RRF와 그래프 후보 보완 결과를 비교합니다.


In [ ]:
# 이미 받은 문서 목록으로 전문 단독·벡터 단독·RRF 결과를 비교합니다.
ranked_lists = {"전문 검색": fulltext_docs, "벡터 검색": dense_docs, "RRF": fused_docs}
route_comparison = []
for label, docs in ranked_lists.items():
    passed_ids = [doc.metadata["source_id"] for doc in docs
                  if doc.metadata["source_id"] in allowed_ids]
    route_comparison.append({
        "검색 방식": label, "검색 문서 수": len(docs), "관계 후보 수": len(passed_ids),
        "상위 제목": [documents_by_id[source_id]["title"] for source_id in passed_ids[:3]],
    })
# 그래프에서만 찾은 후보를 보완한 뒤의 결과도 비교합니다.
route_comparison.append({
    "검색 방식": "RRF + 그래프 후보 보완", "검색 문서 수": len(fused_docs),
    "관계 후보 수": len(candidate_rows),
    "상위 제목": [documents_by_id[row["source_id"]]["title"] for row in candidate_rows[:3]],
})
display(pd.DataFrame(route_comparison))


### 🖐️ 함께 따라하기: 섬 생존 줄거리와 관계 조건을 합칩니다

`practice_semantic`을 `비행기 추락 사고에서 살아남은 남자가 외딴 무인도에서 홀로 생존하는 영화`로 두세요. 전문 검색·Dense는 청크 30개씩 조회해 `practice_routes`에 담으세요. as_documents로 바꾼 두 목록을 ensemble_retriever로 결합합니다. 관계 경로는 `practice_graph_rows`, 관계 후보 ID로 제한한 뒤 그래프에서만 찾은 후보는 검색 점수 0으로 뒤에 추가합니다. source_id·search_rank·search_score가 있는 최종 목록을 `practice_candidates`에 담으세요.

무인도 생존 질문의 세 검색 결과를 합치고 배우·연도 조건을 만족하는 영화만 남깁니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 🖐️ 함께 따라하기: 관계 경로가 보완한 문서를 찾습니다

이미 조회한 `practice_routes`를 사용합니다. 전문 검색과 벡터 검색의 합집합 ID를 `practice_text_ids`, 관계 검색 ID를 `practice_graph_ids`로 만드세요. 관계 검색에만 있는 문서와 그 원문을 확인하세요. 차집합이 비어 있으면 그 사실을 적고, 후보 수 증가 없이도 관계 조건 확인이 필요한 이유를 설명하세요. 새 임베딩이나 검색 호출은 하지 않습니다.

전문·벡터 검색이 놓치고 관계 검색만 찾은 원문을 확인합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

배우·연도 조건을 지키려면 그래프 가중치를 아주 높이면 충분할까요?

<details><summary>해설 보기</summary>

가중치는 상대 점수이므로 필수 조건을 보장하지 않습니다. 조건을 만족하는 ID 집합으로 후보를 제한해야 합니다.

</details>


## 3. 영화의 PageRank를 검색 순위에 반영합니다

**출연 관계망에서 중심적인 영화에 비중을 주면, 검색 후보의 순위가 어떻게 달라지는지 비교합니다.**

1. **계산:** 전체 배우·영화의 `ACTED_IN` 관계를 양방향으로 보고 PageRank를 계산합니다.
2. **적용:** 앞서 찾은 후보 영화의 PageRank를 검색 순위 점수와 결합합니다.
3. **비교:** 같은 후보에서 PageRank 비중을 0·0.2·0.6으로 바꾸어 순위를 확인합니다.

PageRank는 **관계망의 중요도**입니다. 순위가 오른 영화의 줄거리가 질문에도 맞는지 확인하세요.

<img src="images/03_pagerank_multihop.png" width="1100" alt="전체 배우·영화 그래프에서 PageRank 계산, 검색 후보 영화에 적용, 같은 후보의 순위 비교">

계산용 메모리 그래프를 만드는 작업이 **투영**입니다. DB의 관계 방향은 유지합니다.
[GDS PageRank 공식 문서](https://neo4j.com/docs/graph-data-science/current/algorithms/page-rank/)

출연 그래프에서 영화의 PageRank 점수를 계산하고 원문 ID에 연결합니다.


In [ ]:
# PageRank 계산에 사용할 메모리 그래프의 이름을 정합니다.
projection_name = "movie_cast"
# 같은 이름의 메모리 그래프를 지웁니다. DB의 노드·관계는 유지됩니다.
run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName", name=projection_name)
# 영화·인물의 출연 관계로 PageRank 계산용 메모리 그래프를 만듭니다.
run_cypher("""
    CALL gds.graph.project($name, ['Movie', 'Person'],
        // 계산할 때 출연 관계를 양방향으로 다룹니다.
        {ACTED_IN: {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount
""", name=projection_name)
# 전체 배우·영화 노드의 PageRank를 계산해 pagerank 속성에 저장합니다.
run_cypher("""
    CALL gds.pageRank.write($name, {writeProperty: 'pagerank', dampingFactor: 0.85})
    YIELD nodePropertiesWritten RETURN nodePropertiesWritten
""", name=projection_name)

# 재정렬에는 배우 점수가 아닌, 각 원문이 설명하는 영화의 PageRank를 사용합니다.
pagerank_rows = run_cypher("""
    MATCH (d:Document)-[:ABOUT_MOVIE]->(m:Movie)
    RETURN d.id AS source_id, m.title AS title, m.pagerank AS pagerank
    ORDER BY pagerank DESC
""")
# source_id로 해당 영화의 PageRank 점수를 찾을 수 있게 만듭니다.
pagerank_by_id = {row["source_id"]: row["pagerank"] for row in pagerank_rows}
display(pd.DataFrame(pagerank_rows).head())


검색 순위 점수와 포화 함수로 변환한 PageRank를 합쳐 영화 순위를 정합니다.


In [ ]:
from statistics import median

# 전체 영화의 중앙값을 고정 기준으로 사용합니다.
pagerank_pivot = median(pagerank_by_id.values())


def rerank_candidates(rows, weight):
    """검색 순위 점수와 포화 함수로 변환한 영화 PageRank를 결합합니다.

    Args:
        rows: 관계 조건을 통과한 통합 후보.
        weight: PageRank 비중. 0이면 검색 순서를 유지합니다.
    Returns:
        pagerank_score와 final_score를 추가해 정렬한 목록.
    """
    ranked = []
    for row in rows:
        pagerank = pagerank_by_id[row["source_id"]]
        # 기준값에서 0.5이며, 값이 커질수록 1에 천천히 가까워집니다.
        pagerank_score = pagerank / (pagerank + pagerank_pivot)
        # 검색 점수는 이미 0~1인 순위 역수이므로 그대로 사용합니다.
        ranked.append({**row, "pagerank_score": pagerank_score,
                       "final_score": (1 - weight) * row["search_score"] + weight * pagerank_score})
    # 빈 후보는 빈 목록을 반환합니다. 동점이면 검색 순서를 유지합니다.
    return sorted(ranked, key=lambda row: (-row["final_score"],
                                         row["search_rank"] or float("inf"), row["source_id"]))


PageRank 비중을 0·0.2·0.6으로 바꾸며 같은 후보의 순위를 비교합니다.


In [ ]:
# 후보 집합과 검색 점수는 고정하고 PageRank 비중만 비교합니다.
comparison = []
for weight in [0.0, 0.2, 0.6]:
    ranked = rerank_candidates(candidate_rows, weight)
    for rank, row in enumerate(ranked[:3], start=1):
        comparison.append({"pagerank_weight": weight, "rank": rank,
                           "title": documents_by_id[row["source_id"]]["title"], **row})
display(pd.DataFrame(comparison))


### 🖐️ 함께 따라하기: 공유 배우가 많아도 줄거리는 다를 수 있습니다

`practice_candidates`를 그대로 고정해 PageRank 비중 0·0.6의 전체 후보 순위를 비교하세요. `practice_graph_rows`의 공통 배우도 함께 읽습니다. `Cast Away`와 `The Green Mile` 중 섬 생존 조건을 뒷받침하는 원문이 어느 쪽인지 설명하고, 유지할 비중을 적으세요.

PageRank 비중 0·0.6의 순위를 줄거리·공유 배우 근거와 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### ✅ 바로 확인 퀴즈

모든 질문에서 같은 영화가 계속 올라온다면 어느 단계를 확인해야 할까요?

<details><summary>해설 보기</summary>

PageRank 비중과 필수 조건을 확인합니다. 중심성이 관련성을 덮고 있다면 비중을 줄이거나 0으로 두고 원문을 다시 비교합니다.

</details>


## 4. 검색 결과로 답변하고 근거를 확인합니다

**앞서 찾고 정렬한 영화 후보를 바탕으로, 사용자의 질문에 답하는 단계입니다.**

1. **근거 모으기:** 상위 최대 3편의 영화 원문과, 그래프 검색에서 얻은 공통 배우·개봉연도 근거를 모읍니다.
2. **답변 생성:** 질문과 근거를 LLM에 전달하고, 답변에 근거 ID를 인용하게 합니다.
3. **인용 확인:** 답변이 가리킨 원문·출연 관계를 읽어, 설명이 근거와 맞는지 확인합니다.

`build_evidence`는 근거를 모으는 함수이며, `evidence_id`는 답변과 해당 근거를 연결하는 ID입니다.

줄거리·출연·연도 근거를 모으는 함수와 근거를 인용하는 답변 체인을 준비합니다.


In [ ]:
# claim_id로 출연 근거를 찾을 사전을 만듭니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def build_evidence(ranked, graph_rows, top_n=3):
    """상위 영화의 줄거리·개봉연도·출연 근거를 모읍니다."""
    # 후보 원문 ID로 영화 속성과 출연 경로를 찾습니다.
    graph_by_id = {row["source_id"]: row for row in graph_rows}
    evidence = []
    claim_ids = set()

    # 상위 영화의 줄거리와 이미 조회한 제목·개봉연도를 담습니다.
    for row in ranked[:top_n]:
        source_id = row["source_id"]
        doc = documents_by_id[source_id]
        movie = graph_by_id[source_id]
        evidence.extend([
            {"evidence_id": source_id, "kind": "plot", "title": doc["title"],
             "url": doc["url"], "text": doc["text"]},
            {"evidence_id": "movie:" + movie["movie_id"], "kind": "movie_record",
             "text": f"{movie['title']} / 개봉연도: {movie['released']}"},
        ])
        # 양쪽 출연 관계 ID를 집합에 모으면 중복이 자동으로 제거됩니다.
        for path in movie["paths"]:
            claim_ids.update([path["first_claim"], path["second_claim"]])

    # 모아 둔 관계 ID로 실제 출연 근거 문장을 가져옵니다.
    for claim_id in sorted(claim_ids):
        claim = relations_by_id[claim_id]
        evidence.append({"evidence_id": claim_id, "kind": "cast_relation",
                         "text": claim["evidence"]})
    return evidence


# 줄거리·연도·출연 근거를 인용해 답하도록 프롬프트를 만듭니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공한 영화 데이터만 사용해 한국어로 답하세요. 줄거리와 배우·연도 조건을 모두 확인하세요. "
               "답변은 세 항목으로 작성하세요. 1. 작품과 개봉연도, 2. 기준 영화와 공유하는 배우, 3. 질문에 맞는 줄거리. "
               "각 항목의 문장 바로 뒤에 실제 evidence_id를 인용하세요. 대괄호 하나에는 ID 하나만 넣으세요. 여러 ID는 [ID1][ID2] 형태로 붙이고 쉼표로 묶지 마세요. "
               "연도에는 movie_record, 줄거리에는 plot, 공유 배우마다 양쪽 cast_relation의 ID를 인용하세요. "
               "예: - 작품과 연도: 작품명과 연도 [해당 영화 속성 ID]. "
               "evidence_ids 목록에만 ID를 적는 것은 충분하지 않습니다. answer 본문에도 반드시 표시하세요. "
               "배우 본인이 사고를 겪었다고 쓰지 말고 배우와 극중 인물을 구분하세요. "
               "조건은 맞지만 줄거리가 다른 후보를 선택하지 마세요. 부족한 근거는 부족하다고 말하세요. "
               "evidence_ids에는 answer 본문에 인용한 모든 ID를 빠짐없이 한 번씩 담으세요. ID마다 대괄호를 따로 쓰세요. 질문의 줄거리 조건에 맞는 영화가 없으면 유사한 다른 영화를 정답으로 제시하지 마세요. 각 항목은 한 문장으로 쓰고, 줄거리는 질문의 조건에 해당하는 핵심 사건만 요약하세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
# 답변 본문과 인용 ID 목록을 구조화합니다.
class GroundedAnswer(BaseModel):
    answer: str = Field(description="작품·연도, 공유 배우, 줄거리의 세 항목. 각 항목은 한 문장. 본문에 [실제 evidence_id]를 표시하며 대괄호 하나에 ID 하나만 넣은 한국어 답변")
    evidence_ids: list[str] = Field(description="answer 본문의 모든 대괄호 ID를 빠짐없이 담은 목록. 줄거리의 plot 원문 ID, 영화 속성의 movie_record ID, 공유 배우 양쪽 cast_relation ID를 모두 포함합니다.")


# 답변 체인을 구성합니다. invoke에서 실제 호출합니다.
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)


상위 후보의 줄거리·출연·연도 근거를 모아 종류와 ID를 확인합니다.


In [ ]:
# PageRank 비중을 0.2로 정해 답변에 사용할 후보 순서를 만듭니다.
final_ranked = rerank_candidates(candidate_rows, weight=0.2)
evidence = build_evidence(final_ranked, graph_rows, top_n=3)
display(pd.DataFrame(evidence, columns=["evidence_id", "kind"]))
print("전달 근거 글자 수:", sum(len(row["text"]) for row in evidence))


질문과 근거를 모델에 전달해 답변과 인용 ID를 생성합니다.


In [ ]:
# 질문과 근거를 전달해 답변을 생성합니다.
answer = answer_chain.invoke({"question": question, "context": json.dumps(evidence, ensure_ascii=False)})
print(answer.answer)
print("인용 ID:", answer.evidence_ids)


인용 ID의 존재 여부와 본문 표시를 확인하고 인용 원문을 읽습니다.


In [ ]:
# 답변이 인용한 ID로 실제 근거를 찾을 사전을 만듭니다.
evidence_by_id = {row["evidence_id"]: row for row in evidence}
# 본문과 모델 목록 양쪽의 인용 ID를 확인합니다.
inline_ids = set(re.findall(r"\[([^\[\]]+)\]", answer.answer))
unknown_ids = (inline_ids | set(answer.evidence_ids)) - set(evidence_by_id)
# ^는 본문과 인용 목록 중 한쪽에만 있는 근거 ID를 고릅니다.
citation_gap = inline_ids ^ set(answer.evidence_ids)
print("본문과 목록에 인용 있음:", bool(inline_ids) and bool(answer.evidence_ids))
print("제공하지 않은 인용 ID:", unknown_ids)
print("본문과 목록의 차이:", citation_gap)
for evidence_id in answer.evidence_ids:
    if evidence_id in evidence_by_id:
        item = evidence_by_id[evidence_id]
        print("\n근거 ID:", evidence_id, "| 종류:", item["kind"])
        print(item["text"])


### 🖐️ 함께 따라하기: 섬 생존 질문의 답변과 인용을 대조합니다

`practice_candidates`를 비중 0으로 재정렬하고 상위 3편의 근거를 `practice_evidence`에 모으세요. Apollo 13·공유 배우·1999~2000·섬 생존을 모두 포함한 한국어 질문으로 답변을 한 번 만드세요. Cast Away의 줄거리, Tom Hanks의 양쪽 출연 관계, 개봉연도 근거를 각각 찾아 읽습니다.

무인도 생존 질문에 답변을 생성하고 실제 인용 근거와 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 결과가 부족하면 어느 단계를 고칠까요?

| 관찰 결과 | 먼저 확인할 곳 |
|---|---|
| 필요한 영화가 모든 후보에 없음 | 가져올 청크 수·검색 언어·생성 Cypher의 조건 |
| 통합 후보에는 있지만 조건 적용 뒤 사라짐 | 기준 영화·연도 경계·공유 배우 수 |
| 후보에는 있지만 최종 상위에서 밀림 | 경로 가중치·PageRank 비중·최종 문서 수 |
| 영화는 맞지만 배우나 연도 설명이 틀림 | 관계 ID·영화 속성·LLM 인용 |
| 인용 ID는 있으나 주장과 관계없음 | 인용 원문이 실제 주장을 뒷받침하는지 |

한 번에 하나의 설정만 바꾸고 같은 질문·후보·최종 개수를 유지해 비교하세요.


### ✅ 바로 확인 퀴즈

후보에 없는 영화를 PageRank 재정렬이나 답변 프롬프트 수정으로 되찾을 수 있나요?

<details><summary>해설 보기</summary>

재정렬은 현재 후보의 순서만 바꿉니다. 후보 검색·관계 조건·확장 한도에서 누락이 시작된 곳을 먼저 고쳐야 합니다.

</details>


## 이번 강의 정리

- 세 경로의 반환 단위를 원문 ID로 통일하고, 경로 내부 중복을 제거했습니다.
- 전문·벡터 검색의 RRF 순위와 그래프의 후보·근거 역할을 구분했습니다.
- 조건 통과 후보를 고정한 뒤 PageRank의 영향을 비교했습니다.
- 출연 경로·줄거리 원문·개봉연도 근거를 나누어 답변과 인용을 대조했습니다.


## 선택 실습: 새 질문으로 같은 흐름을 실행합니다

**1~4절에서 만든 검색·재정렬·답변 흐름을 다른 질문에 적용하는 추가 연습입니다.**

> Sleepless in Seattle과 출연 배우를 최소 2명 공유하고 1993년 이후 개봉한 영화 중,
> 서로 경쟁하는 서점 주인들이 익명 이메일로 사랑하게 되는 작품을 찾아 주세요.

`You've Got Mail`의 원문과 `Tom Hanks`·`Meg Ryan`의 출연 관계가 판단 근거입니다.
`Joe Versus the Volcano`는 공유 배우만 보고 포함하면 안 됩니다. 개봉연도 조건도 확인하세요.
이 정보는 순위 결과가 아니라 **기존 데이터에서 확인할 기준**입니다.


### 🖐️ 함께 따라하기: 같은 통합 검색을 새 질문에 적용합니다

줄거리 검색어는 `경쟁하는 서점 주인들이 서로의 정체를 모른 채 익명 이메일로 사랑하게 되는 영화`를 사용합니다. 기준 영화 `Sleepless in Seattle`, 1993년 이후 개봉, 최소 공유 배우 2명 조건을 자연어로 질문하세요. 관계 체인·전문 검색·Dense·융합·필수 조건·PageRank 비중 0·근거 조립까지 연결하고, 각 단계의 고유 문서 수를 출력하고 `new_evidence`에 근거를 모으세요. 이 셀에서는 원문까지 확인합니다.

새 질문으로 검색한 결과를 합치고 배우 수·연도 조건에 맞는 상위 영화의 근거를 모읍니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


### 원문을 읽은 뒤 답변을 요청합니다

위 결과의 줄거리·공통 배우·개봉연도를 확인한 뒤 실행합니다.
`new_question`에는 위의 한국어 질문 전체를 넣고, `new_evidence`를 전달해 `new_answer`를 만드세요.
모델을 한 번 호출하고 답변과 실제 인용 ID를 읽습니다.

익명 이메일 영화 질문의 답변을 생성하고 인용 ID와 근거를 대조합니다.


In [ ]:
# 지문에 따라 코드를 작성하세요.


## 교안 02 핵심 코드 이어서 보기

본문에서 그래프와 인덱스를 적재한 뒤 실행합니다.


DB·모델을 연결하고 답변에 사용할 원문·관계 근거를 불러옵니다.


In [ ]:
import json
import os
import re
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_typesafe import Choice, TypeSafeClassifier
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.types import RetrieverResultItem
from pydantic import BaseModel, Field

# 교안의 .env로 DB와 질문 임베딩 모델을 연결합니다.
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
# .env의 주소·계정으로 Neo4j에 연결합니다.
neo4j_uri = os.environ["NEO4J_URI"]
driver = GraphDatabase.driver(
    neo4j_uri, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large", dimensions=768,
    check_embedding_ctx_length=False,  # 질문을 자동으로 나누지 않습니다.
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


# 답변에 인용할 원문과 관계 근거를 불러옵니다.
movies = json.loads((material_dir / "data/movies_extraction_packet.json").read_text(encoding="utf-8"))
documents_by_id = movies["documents"]


검색된 청크에서 영화별 대표 후보를 고르는 함수를 만듭니다.


In [ ]:
def format_vector(record):
    """검색된 청크의 본문과 원문 ID·유사도 점수를 반환합니다."""
    node = record["node"]
    # content는 청크 본문, metadata는 청크 ID·원문 ID·유사도를 담는 사전입니다.
    return RetrieverResultItem(content=node["text"], metadata={
        "chunk_id": node["id"], "source_id": node["source_id"], "score": record["score"]
    })


# 벡터 인덱스와 결과 변환 함수를 연결해 벡터 검색기를 만듭니다.
vector_retriever = VectorRetriever(
    driver, index_name="movie_chunks", embedder=embedding_model,
    result_formatter=format_vector,
)


def vector_documents(query_vector, top_k=12):
    """검색된 청크 중 영화별 최고 점수 한 건을 대표 후보로 반환합니다."""
    # 질문 벡터와 유사한 청크를 점수 내림차순으로 검색합니다.
    result = vector_retriever.search(query_vector=query_vector, top_k=top_k)
    # 영화별 후보 목록을 만들기 위해 처음 나온 최고 점수 청크를 대표로 남깁니다.
    rows, seen = [], set()
    for item in result.items:
        row = item.metadata
        if row["source_id"] not in seen:
            seen.add(row["source_id"])
            rows.append(dict(row))
    return rows


질문의 언어에 맞춰 전문 검색을 수행하는 함수를 만듭니다.


In [ ]:
# Jev가 한글·영어 중 질문의 언어를 선택하도록 설정합니다.
language_classifier = TypeSafeClassifier(model=os.getenv("TYPESAFE_MODEL", "jev-1.13.0"))
language_question = Choice(
    instructions="질문의 주된 언어를 선택하세요.",
    criteria={"ko": "한글 질문. 영어 영화명 등이 섞인 한글 문장도 포함합니다.",
              "en": "한글 없이 영어 단어나 문장으로 작성된 질문입니다."},
)


# 영어 검색어를 문자열 목록으로 받습니다.
class EnglishKeywords(BaseModel):
    """영어 원문 검색에 사용할 핵심어 목록입니다."""
    keywords: list[str] = Field(description="중복 없는 영어 단어 4~8개")


# 한글 질문에서 영어 검색어 목록을 만드는 체인입니다.
lexical_prompt = ChatPromptTemplate.from_messages([
    ("system", "한글 영화 줄거리 질문을 영어 원문 검색용 핵심어 4~8개로 바꾸세요. "
               "질문의 핵심 사건·대상을 나타내는 단어를 골라 keywords의 각 항목에 영어 단어 하나씩 담으세요. "
               "영화 정답이나 배우를 추측해 추가하지 마세요. 질문에 없는 고유명사는 넣지 마세요."),
    ("human", "{question}"),
])
lexical_chain = lexical_prompt | llm.with_structured_output(EnglishKeywords)


def fulltext_documents(query, top_k=12):
    """질문의 언어에 맞춰 전문 검색하고 원문별 최고 점수 청크를 반환합니다."""
    # state에 질문, questions에 언어 선택 기준을 전달합니다.
    decision = language_classifier.invoke({
        "state": query, "questions": {"language": language_question},
    })
    language = decision.choices["language"].choice
    # 한글은 영어 검색어 목록을 받아 OR로 연결하고, 영어는 그대로 사용합니다.
    if language == "ko":
        terms = lexical_chain.invoke({"question": query})
        search_text = " OR ".join(f'"{word}"' for word in terms.keywords)
    else:
        search_text = query
    # 전문 인덱스에서 검색어와 일치하는 청크를 최대 top_k개 조회합니다.
    hits = run_cypher("""
        // 전문 인덱스에서 관련 청크를 최대 top_k개 조회합니다.
        CALL db.index.fulltext.queryNodes('movie_fulltext', $search_text, {limit: $top_k})
        YIELD node, score
        // 결과 통합에 쓸 원문 ID를 반환합니다.
        RETURN node.source_id AS source_id, node.id AS chunk_id, score
        // 점수 내림차순으로 정렬합니다.
        ORDER BY score DESC, chunk_id
    """, search_text=search_text, top_k=top_k)
    rows, seen = [], set()
    # 영화별 순위를 비교하도록 원문마다 최고 점수 검색 행을 대표로 남깁니다.
    for row in hits:
        source_id = row["source_id"]
        if source_id not in seen:
            seen.add(source_id)
            rows.append(row)
    return rows


관계 질문을 Cypher로 바꿔 조회하는 체인을 만듭니다.


In [ ]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph

# 앞에서 설정한 실습 DB 계정으로 연결합니다.
graph = Neo4jGraph(
    url=neo4j_uri,
    username=os.environ["NEO4J_USER"],
    password=os.environ["NEO4J_PASSWORD"],
    enhanced_schema=False,  # 속성 예시 값 수집을 생략하고 기본 스키마를 사용합니다.
)
# 관계 질문을 지정한 스키마의 조회 Cypher로 바꾸는 규칙을 정합니다.
# GraphCypherQAChain이 {schema}에는 DB에서 읽어 include_types로 추린 구조를,
# {question}에는 invoke에 전달한 질문을 자동으로 넣습니다.
cypher_prompt = PromptTemplate.from_template("""
다음 스키마만 사용해 조회용 Cypher 하나를 작성하세요.
{schema}

사용할 관계 경로:
(base:Movie)<-[first:ACTED_IN]-(actor:Person)-[second:ACTED_IN]->(candidate:Movie)<-[:ABOUT_MOVIE]-(d:Document)

기준 영화명·개봉연도·최소 공유 배우 수는 질문에서 읽으세요.
한글과 영문 영화명이 함께 있으면 영문 title을 사용하세요.
기준 영화명은 base.title에, 개봉연도 조건은 candidate.released에만 적용하세요.
다른 영화를 요구하면 candidate <> base로 기준 영화를 제외하세요.
이 조건들은 집계 WITH 앞의 WHERE에서 적용하세요.

공유 배우와 두 출연 관계의 실제 claim_id는 다음처럼 함께 집계하세요.
WITH candidate, d, count(DISTINCT actor) AS shared_count,
     collect(DISTINCT {{actor: actor.name, first_claim: first.claim_id, second_claim: second.claim_id}}) AS paths
공유 배우 수 조건은 이 WITH 뒤의 WHERE에서 적용하세요.

반환 형식:
RETURN d.id AS source_id, candidate.title AS title, candidate.released AS released,
       candidate.standard_id AS movie_id, shared_count, paths
ORDER BY source_id LIMIT 100

질문에 없는 조건이나 줄거리 속성을 추가하지 마세요. Cypher만 반환하세요.
질문: {question}
""")
# 질문 → Cypher 생성 → DB 조회를 연결합니다.
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, cypher_prompt=cypher_prompt,
    # 필요한 레이블·관계 유형만 프롬프트에 넣습니다.
    include_types=["Movie", "Person", "Document", "ACTED_IN", "ABOUT_MOVIE"],
    return_direct=True,  # LLM 답변 생성 없이 DB 조회 결과를 반환합니다.
    return_intermediate_steps=False,  # 중간 실행 정보는 반환하지 않습니다.
    top_k=100,  # 조회 결과를 최대 100행 사용합니다.
    # LLM 생성 쿼리의 실행 위험에 동의합니다. False이면 체인 생성이 차단됩니다.
    allow_dangerous_requests=True,
)


EnsembleRetriever에 전문·벡터 검색을 연결합니다.


In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_classic.retrievers import EnsembleRetriever


def as_documents(rows):
    """원문별 검색 결과를 LangChain Document 목록으로 바꿉니다."""
    return [Document(page_content=documents_by_id[row["source_id"]]["text"],
                     metadata={"source_id": row["source_id"]}) for row in rows]


# 기존 검색 함수를 LangChain에서 호출할 수 있도록 연결합니다.
# 두 함수는 이미 원문별 중복을 제거한 결과를 반환합니다.
fulltext_retriever = RunnableLambda(
    lambda query: as_documents(fulltext_documents(query, top_k=30))
)
dense_retriever = RunnableLambda(
    lambda query: as_documents(vector_documents(embedding_model.embed_query(query), top_k=30))
)
# 전문·벡터 검색만 가중 RRF로 결합합니다. 같은 원문은 source_id로 합칩니다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[fulltext_retriever, dense_retriever],
    weights=[0.4, 0.6], c=60, id_key="source_id",
)


영화의 PageRank 점수를 계산합니다.


In [ ]:
# PageRank 계산에 사용할 메모리 그래프의 이름을 정합니다.
projection_name = "movie_cast"
# 같은 이름의 메모리 그래프를 지웁니다. DB의 노드·관계는 유지됩니다.
run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName", name=projection_name)
# 영화·인물의 출연 관계로 PageRank 계산용 메모리 그래프를 만듭니다.
run_cypher("""
    CALL gds.graph.project($name, ['Movie', 'Person'],
        // 계산할 때 출연 관계를 양방향으로 다룹니다.
        {ACTED_IN: {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount
""", name=projection_name)
# 전체 배우·영화 노드의 PageRank를 계산해 pagerank 속성에 저장합니다.
run_cypher("""
    CALL gds.pageRank.write($name, {writeProperty: 'pagerank', dampingFactor: 0.85})
    YIELD nodePropertiesWritten RETURN nodePropertiesWritten
""", name=projection_name)

# 재정렬에는 배우 점수가 아닌, 각 원문이 설명하는 영화의 PageRank를 사용합니다.
pagerank_rows = run_cypher("""
    MATCH (d:Document)-[:ABOUT_MOVIE]->(m:Movie)
    RETURN d.id AS source_id, m.title AS title, m.pagerank AS pagerank
    ORDER BY pagerank DESC
""")
# source_id로 해당 영화의 PageRank 점수를 찾을 수 있게 만듭니다.
pagerank_by_id = {row["source_id"]: row["pagerank"] for row in pagerank_rows}


전체 영화의 중앙값을 기준으로 PageRank를 포화 변환하고 검색 순위 점수와 결합합니다.


In [ ]:
from statistics import median

# 전체 영화의 중앙값을 고정 기준으로 사용합니다.
pagerank_pivot = median(pagerank_by_id.values())


def rerank_candidates(rows, weight):
    """검색 순위 점수와 포화 함수로 변환한 영화 PageRank를 결합합니다.

    Args:
        rows: 관계 조건을 통과한 통합 후보.
        weight: PageRank 비중. 0이면 검색 순서를 유지합니다.
    Returns:
        pagerank_score와 final_score를 추가해 정렬한 목록.
    """
    ranked = []
    for row in rows:
        pagerank = pagerank_by_id[row["source_id"]]
        # 기준값에서 0.5이며, 값이 커질수록 1에 천천히 가까워집니다.
        pagerank_score = pagerank / (pagerank + pagerank_pivot)
        # 검색 점수는 이미 0~1인 순위 역수이므로 그대로 사용합니다.
        ranked.append({**row, "pagerank_score": pagerank_score,
                       "final_score": (1 - weight) * row["search_score"] + weight * pagerank_score})
    # 빈 후보는 빈 목록을 반환합니다. 동점이면 검색 순서를 유지합니다.
    return sorted(ranked, key=lambda row: (-row["final_score"],
                                         row["search_rank"] or float("inf"), row["source_id"]))


근거를 모으는 함수와 근거를 인용하는 답변 체인을 만듭니다.


In [ ]:
# claim_id로 출연 근거를 찾을 사전을 만듭니다.
relations_by_id = {row["claim_id"]: row for row in movies["relations"]}


def build_evidence(ranked, graph_rows, top_n=3):
    """상위 영화의 줄거리·개봉연도·출연 근거를 모읍니다."""
    # 후보 원문 ID로 영화 속성과 출연 경로를 찾습니다.
    graph_by_id = {row["source_id"]: row for row in graph_rows}
    evidence = []
    claim_ids = set()

    # 상위 영화의 줄거리와 이미 조회한 제목·개봉연도를 담습니다.
    for row in ranked[:top_n]:
        source_id = row["source_id"]
        doc = documents_by_id[source_id]
        movie = graph_by_id[source_id]
        evidence.extend([
            {"evidence_id": source_id, "kind": "plot", "title": doc["title"],
             "url": doc["url"], "text": doc["text"]},
            {"evidence_id": "movie:" + movie["movie_id"], "kind": "movie_record",
             "text": f"{movie['title']} / 개봉연도: {movie['released']}"},
        ])
        # 양쪽 출연 관계 ID를 집합에 모으면 중복이 자동으로 제거됩니다.
        for path in movie["paths"]:
            claim_ids.update([path["first_claim"], path["second_claim"]])

    # 모아 둔 관계 ID로 실제 출연 근거 문장을 가져옵니다.
    for claim_id in sorted(claim_ids):
        claim = relations_by_id[claim_id]
        evidence.append({"evidence_id": claim_id, "kind": "cast_relation",
                         "text": claim["evidence"]})
    return evidence


# 줄거리·연도·출연 근거를 인용해 답하도록 프롬프트를 만듭니다.
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공한 영화 데이터만 사용해 한국어로 답하세요. 줄거리와 배우·연도 조건을 모두 확인하세요. "
               "답변은 세 항목으로 작성하세요. 1. 작품과 개봉연도, 2. 기준 영화와 공유하는 배우, 3. 질문에 맞는 줄거리. "
               "각 항목의 문장 바로 뒤에 실제 evidence_id를 인용하세요. 대괄호 하나에는 ID 하나만 넣으세요. 여러 ID는 [ID1][ID2] 형태로 붙이고 쉼표로 묶지 마세요. "
               "연도에는 movie_record, 줄거리에는 plot, 공유 배우마다 양쪽 cast_relation의 ID를 인용하세요. "
               "예: - 작품과 연도: 작품명과 연도 [해당 영화 속성 ID]. "
               "evidence_ids 목록에만 ID를 적는 것은 충분하지 않습니다. answer 본문에도 반드시 표시하세요. "
               "배우 본인이 사고를 겪었다고 쓰지 말고 배우와 극중 인물을 구분하세요. "
               "조건은 맞지만 줄거리가 다른 후보를 선택하지 마세요. 부족한 근거는 부족하다고 말하세요. "
               "evidence_ids에는 answer 본문에 인용한 모든 ID를 빠짐없이 한 번씩 담으세요. ID마다 대괄호를 따로 쓰세요. 질문의 줄거리 조건에 맞는 영화가 없으면 유사한 다른 영화를 정답으로 제시하지 마세요. 각 항목은 한 문장으로 쓰고, 줄거리는 질문의 조건에 해당하는 핵심 사건만 요약하세요."),
    ("human", "질문: {question}\n근거: {context}"),
])
# 답변 본문과 인용 ID 목록을 구조화합니다.
class GroundedAnswer(BaseModel):
    answer: str = Field(description="작품·연도, 공유 배우, 줄거리의 세 항목. 각 항목은 한 문장. 본문에 [실제 evidence_id]를 표시하며 대괄호 하나에 ID 하나만 넣은 한국어 답변")
    evidence_ids: list[str] = Field(description="answer 본문의 모든 대괄호 ID를 빠짐없이 담은 목록. 줄거리의 plot 원문 ID, 영화 속성의 movie_record ID, 공유 배우 양쪽 cast_relation ID를 모두 포함합니다.")


# 답변 체인을 구성합니다. invoke에서 실제 호출합니다.
answer_chain = answer_prompt | llm.with_structured_output(GroundedAnswer)


세 검색 결과를 통합하고 배우·연도 조건을 만족하는 영화만 남깁니다.


In [ ]:
question = "The Matrix와 배우를 공유하고 1999년 이전 개봉한 다른 영화 중, 뇌에 기밀 데이터를 저장·운반하다 위험에 빠지는 작품은 무엇인가요?"
semantic_query = "기밀 정보를 뇌 속 장치에 저장해 운반하는 사람이 위험에 빠지는 영화"
graph_request = "매트릭스(The Matrix)와 배우를 최소 한 명 공유하며 1999년 이전에 개봉한 다른 영화를 모두 찾아 주세요."

# 질문 벡터와 관계 검색 결과를 한 번씩 구합니다.
query_vector = embedding_model.embed_query(semantic_query)
graph_result = graph_chain.invoke({"query": graph_request})
graph_rows = graph_result["result"]
# Text-to-Cypher가 반환한 후보 ID를 필수 관계 조건에 사용합니다.
allowed_ids = {row["source_id"] for row in graph_rows}
fulltext_docs = fulltext_retriever.invoke(semantic_query)
dense_docs = as_documents(vector_documents(query_vector, top_k=30))
fused_docs = ensemble_retriever.weighted_reciprocal_rank([fulltext_docs, dense_docs])
fused_rows = [{"source_id": doc.metadata["source_id"], "search_rank": rank, "search_score": 1 / rank}
              for rank, doc in enumerate(fused_docs, start=1)]
# 관계 조건 후보만 남기고, 검색에서 빠진 그래프 후보도 보관합니다.
candidate_rows = [row for row in fused_rows if row["source_id"] in allowed_ids]
fused_ids = {row["source_id"] for row in fused_rows}
candidate_rows += [{"source_id": source_id, "search_rank": None, "search_score": 0.0}
                   for source_id in sorted(allowed_ids - fused_ids)]


후보를 재정렬하고 근거를 인용한 답변을 생성합니다.


In [ ]:
# PageRank 비중 0.2로 정렬한 상위 3편의 근거를 모읍니다.
final_ranked = rerank_candidates(candidate_rows, weight=0.2)
evidence = build_evidence(final_ranked, graph_rows, top_n=3)
answer = answer_chain.invoke({"question": question, "context": json.dumps(evidence, ensure_ascii=False)})
print(answer.answer)
